In [ ]:
# INSTALL IF NECESSARY

%pip install pandas numpy pillow matplotlib tensorflow scikit-learn joblib

In [ ]:
# -*- coding: utf-8 -*-

**AER850 Project Phase II: Data Preprocessing**


Mars Surface / Curiosity image set (Kaggle)

In [ ]:
# SETUP
from pathlib import Path
import json  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()  # finds current directory
DATA_DIR = PROJECT_ROOT / "Data" / "Mars_Surface_and_Curiosity_Image"
IMAGE_DIR = DATA_DIR / "images"  # complete image collection location
OUT_DIR = PROJECT_ROOT / "processed"  # output will be saved here

# every image gets resized to IMG_SIZE x IMG_SIZE
IMG_SIZE = 224

BATCH_SIZE = 16  # process a few images at a time; can lower this if memory is limited
SEED = 42   # allows for reproducibility of random operations

# error messages for missing directories
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Missing images folder: {IMAGE_DIR}")
OUT_DIR.mkdir(exist_ok=True)

# check that directories are correct
print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Generated files:", OUT_DIR)

Files are split into training, validation, and test based on the .csv's that are included with the dataset.


In [ ]:
# load the CSV's

split_files = { 
    "train": "Train_CSV.csv",
    "validation": "Validation_CSV.csv",
    "test": "Test_CSV.csv",
}

tables = []  # empty list that will hold three tables
for split, filename in split_files.items():
    table = pd.read_csv(DATA_DIR / filename, dtype={"JPG": "string", "LABELS": "Int64"})
    table = table.rename(columns={"JPG": "file", "LABELS": "label"})
    table["split"] = split  # add the split name to every row in this table
    tables.append(table)  # add this table to the list
df = pd.concat(tables, ignore_index=True)  # stack the tables and number rows from zero
display(df.head(10))  # show the first ten rows
display(df["split"].value_counts())  # count rows in each split

The dataset is encoded using integer labels; each image category has a number defined in Labels Information.txt.

The labels are defined as shown below:

| ID | Target / Feature | ID | Target / Feature | ID | Target / Feature |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | apxs | **9** | horizon | **18** | portion tube opening |
| **1** | apxs cal target | **10** | inlet | **19** | rems uv sensor |
| **2** | chemcam cal target | **11** | mahli | **20** | rover rear deck |
| **3** | chemin inlet open | **12** | mahli cal target | **21** | scoop |
| **4** | drill | **13** | mastcam | **22** | sun |
| **5** | drill holes | **14** | mastcam cal target | **23** | turret |
| **6** | drt front | **15** | observation tray | **24** | wheel |
| **7** | drt side | **16** | portion box | | |
| **8** | ground | **17** | portion tube | | |


**Notes**: Sun (22) has no examples anywhere, several classes are absent from validation or test, and class proportions differ substantially between splits.


For this reason, new stratified splits will be created. 70% will be used to train, and the remaining 30% is split evenly between validation and test sets. The combined data from all 3 CSV's will be used for this, so that the labels stay attached to their respective images.

In [ ]:
# keep original assignment for comparison
original_df = df.copy()

# 70% training
new_train, held_out = train_test_split(
    original_df,
    test_size=0.30,
    stratify=original_df["label"],
    random_state=42
)

# 15% validation, 15% test
new_validation, new_test = train_test_split(
    held_out,
    test_size=0.50,
    stratify=held_out["label"],
    random_state=42
)

# record new assignments without modifying original CSVs
new_train = new_train.assign(split="train")
new_validation = new_validation.assign(split="validation")
new_test = new_test.assign(split="test")

df = pd.concat(
    [new_train, new_validation, new_test],
    ignore_index=True
)

display(df.head(10))  # show the first ten rows
display(df["split"].value_counts())  # count rows in each split

Comparison of label percentages in train, validation, and data splits.

In [ ]:
id_to_name = {}  # maps numeric ID to class name
with (DATA_DIR / "Labels Information.txt").open(encoding="utf-8-sig") as handle:
    for line in handle:
        if line.strip():
            label_id, class_name = line.strip().split(maxsplit=1)
            id_to_name[int(label_id)] = class_name

def summarize_labels(data):
    # count each label in each set
    counts = pd.crosstab(data["label"], data["split"])

    # include all defined labels, even those with 0 images
    counts = counts.reindex(
        index=sorted(id_to_name),
        columns=["train", "validation", "test"],
        fill_value=0
    )

    # summary table with class names
    summary = pd.DataFrame(index=counts.index)
    summary["class_name"] = summary.index.map(id_to_name)

    # add count and percentage for each set
    for split in ["train", "validation", "test"]:
        summary[f"{split}_count"] = counts[split]

        summary[f"{split}_percent"] = (
            100 * counts[split] / counts[split].sum()
        ).round(2)

    return summary.rename_axis("label").reset_index()


# create both summaries (ogirinal & resplit)
original_summary = summarize_labels(original_df)
new_summary = summarize_labels(df)

# display both tables
print("ORIGINAL SPLITS — percentages within each set")
display(original_summary)
print("NEW STRATIFIED SPLITS — percentages within each set")
display(new_summary)

# combine both summaries into one CSV
original_summary.insert(0, "split_strategy", "original")
new_summary.insert(0, "split_strategy", "stratified")
comparison = pd.concat(
    [original_summary, new_summary],
    ignore_index=True
)

OUT_DIR.mkdir(exist_ok=True)
output_path = OUT_DIR / "original_vs_stratified_class_percentages.csv"
comparison.to_csv(output_path, index=False)

print("Saved to:", output_path)

Check for repeated filenames between splits (this does not detect duplicate image content, mislabeled visual content or similar issues, which require additional inspection).

**There are 6691 image files across all splits.**

In [ ]:
df["filename"] = df["file"].str.replace("\\", "/", regex=False).str.rsplit("/", n=1).str[-1]
duplicates = df[df["filename"].duplicated(keep=False)]
if not duplicates.empty:
    display(duplicates.sort_values("filename"))
    raise ValueError("Repeated filename found. Investigate within-split duplicates and split overlap.")
print("Unique filenames:", df["filename"].nunique())
print("No filename overlap between the supplied splits.")


Check to see whether any image files have errors, with the list based on the provided split CSV's.

The audit checks:
- Does the image file exist?
- Can Python open and read its pixels?
- What are its width and height?
- Is it grayscale (L) or colour (RGB)?

Outputs a CSV; any flagged errors should be checked and any issues resolved before running feature extraction.

In [ ]:
checks = []

for reference in df["file"]:
    path = IMAGE_DIR / Path(reference.replace("\\", "/")).name

    result = {
        "image_relpath": "",
        "width": None,
        "height": None,
        "mode": "",
        "error": ""
    }

    try:
        with Image.open(path) as im:
            im.load()
            result.update(
                image_relpath=path.relative_to(PROJECT_ROOT).as_posix(),
                width=im.width,
                height=im.height,
                mode=im.mode
            )
    except (OSError, ValueError) as error:
        result["error"] = str(error)

    checks.append(result)

# replaces previous results so this cell can be rerun
audit = pd.DataFrame(checks, index=df.index)
df = df.drop(columns=audit.columns, errors="ignore")
df[audit.columns] = audit

# save the CSV and print the number of problems
df.to_csv(OUT_DIR / "image_audit.csv", index=False)
problem_count = (df["error"] != "").sum()
print(f"Image checks complete: {problem_count} problems found. Saved image_audit.csv.")

Based on the CSV generated from the above cell, the images have multiple different dimensions. The image size must be standardized across the entire dataset.

- The dataset contains both RGB and B&W (L) images; all images will be converted to RGB.
- Because the image will be changed into 8-bit RGB, all values will be in a range of 0-255.

`load_pixels` prepares the image:
- Makes it three-channel RGB. Each pixel has red, green, and blue values, and a grayscale image gets its brightness copied into all three channels.
- Makes it 224 × 224 pixels. Resizes without stretching the scene (to avoid distortion of image elements), then adds black borders where needed (padding).
- Turns it into numbers. The result has the shape (224, 224, 3): height, width, & 3 colour channels.
- Uses floating-point numbers. float32 allows calculations with decimals. Changing 120 to 120.0 does not change the brightness or scale (values are still between 0 and 255).

All of the processed images are saved into processed/images

In [ ]:
def load_pixels(relative_path):
    with Image.open(PROJECT_ROOT / relative_path) as im:
        # convert colour & grayscale images to 3-channel RGB.
        rgb = im.convert("RGB")

        # resize without stretching, adding black padding if needed
        resized = ImageOps.pad(
            rgb,
            (IMG_SIZE, IMG_SIZE),
            method=Image.Resampling.BILINEAR,
            color=(0, 0, 0)
        )

        # return floating-point pixels with values 0–255
        return np.asarray(resized, dtype=np.float32)


# create folder for processed images
processed_image_dir = OUT_DIR / "images"
processed_image_dir.mkdir(parents=True, exist_ok=True)

processed_paths = []

for relative_path in df["image_relpath"]:
    pixels = load_pixels(relative_path)

    # save a resized RGB copy as an 8-bit PNG
    output_path = processed_image_dir / (
        Path(relative_path).stem + ".png"
    )
    Image.fromarray(pixels.astype(np.uint8)).save(output_path)

    # record its location relative to the project root folder
    processed_paths.append(
        output_path.relative_to(PROJECT_ROOT).as_posix()
    )

df["processed_image_relpath"] = processed_paths
df.to_csv(OUT_DIR / "processed_image_data.csv", index=False)

print(f"Saved {len(processed_paths)} processed images.")

Images must be prepared for ResNet50:
- Change the channel order from red, green, blue to blue, green, red.
- Subtract a fixed average value from each channel, based on the ImageNet images used to train the model.